In [107]:
# House Prices - Advanced Regression Techniques
## Predict sales prices and practice feature engineering, RFs, and gradient boosting

In [108]:
import kagglehub

path = kagglehub.competition_download('house-prices-advanced-regression-techniques')
kagglehub.whoami()

Kaggle credentials successfully validated.


{'username': 'seradrug'}

In [109]:
# Imports 

In [161]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error

In [162]:
# EDA

In [163]:
df_train = pd.read_csv(os.path.join(path, "train.csv"))
df_test = pd.read_csv(os.path.join(path, "test.csv"))
df_train.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [164]:
lapse_ord = [
    "Utilities", # str -> ordinal
    "BsmtQual", # str -> ordinal
    "BsmtCond", # str -> ordinal 
    "BsmtExposure", # str -> ordinal
    "FireplaceQu", # str -> ordinal 
    "GarageFinish", # str -> ordinal
    "GarageQual", # str -> ordinal
    "GarageCond", # str -> ordinal
    "PoolQC", # str -> ordinal
    "Fence", # str -> ordinal
    "BsmtFinType1", # str -> ordinal
    "BsmtFinType2", # str -> ordinal
    "KitchenQual", # str -> ordinal
    "Functional", # str -> ordinal
]

In [165]:
lapse_ord_nan = [
    "BsmtQual", # str -> ordinal
    "BsmtCond", # str -> ordinal 
    "BsmtExposure", # str -> ordinal
    "FireplaceQu", # str -> ordinal 
    "GarageFinish", # str -> ordinal
    "GarageQual", # str -> ordinal
    "GarageCond", # str -> ordinal
    "PoolQC", # str -> ordinal
    "Fence", # str -> ordinal
    "BsmtFinType1", # str -> ordinal
    "BsmtFinType2", # str -> ordinal
]

In [166]:
lapse_ord_mode = [
    "Utilities", # str -> ordinal
    "KitchenQual", # str -> ordinal
    "Functional", # str -> ordinal
]

In [167]:
lapse_int = [
    "LotFrontage", # int
    "MasVnrArea", # int
    "GarageYrBlt", # int 
    "BsmtFinSF1", # int
    "BsmtFinSF2", # int
    "BsmtUnfSF", # int
    "TotalBsmtSF", # int
    "BsmtFullBath", # int
    "BsmtHalfBath", # int
    "GarageCars", # int
    "GarageArea", # int
]

In [168]:
lapse_cat = [
    "Alley", # str -> onehot
    "MasVnrType", # str -> onehot
    "Electrical", # str -> onehot
    "GarageType", # str -> onehot
    "MiscFeature", # str -> onehot
    "Exterior1st", # str -> onehot
    "Exterior2nd", # str -> onehot
    "SaleType", # str -> onehot
    "MSZoning", # str -> onehot
]

In [169]:
ordinal_categories = [
    # Utilities
    ["ELO", "NoSeWa", "NoSewr", "AllPub"],

    # BsmtQual
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],

    # BsmtCond
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],

    # BsmtExposure
    ["None", "No", "Mn", "Av", "Gd"],

    # FireplaceQu
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],

    # GarageFinish
    ["None", "Unf", "RFn", "Fin"],

    # GarageQual
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],

    # GarageCond
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],

    # PoolQC
    ["None", "Fa", "TA", "Gd", "Ex"],

    # Fence
    ["None", "MnWw", "GdWo", "MnPrv", "GdPrv"],

    # BsmtFinType1
    ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],

    # BsmtFinType2
    ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],

    # KitchenQual
    ["Po", "Fa", "TA", "Gd", "Ex"],

    # Functional
    ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"]
]

In [170]:
df_train[lapse_ord_nan] = df_train[lapse_ord_nan].fillna("None")
df_test[lapse_ord_nan] = df_test[lapse_ord_nan].fillna("None")

In [171]:
## Нужно разделить NaN на три случая:
## NaN --> ordinal (заполняем нулём),
## NaN --> int (заполняем средним/медианой/модой) 
## NaN --> onehot (заполняем модой)

In [172]:
for i in lapse_cat:
    df_train[i] = df_train[i].fillna(df_train[i].mode()[0])
    df_test[i] = df_test[i].fillna(df_test[i].mode()[0])

In [173]:
for i in lapse_int:
    df_train[i] = df_train[i].fillna(df_train[i].mean())
    df_test[i] = df_test[i].fillna(df_test[i].mean())

In [174]:
for i in lapse_ord_mode:
    df_train[i] = df_train[i].fillna(df_train[i].mode()[0])
    df_test[i] = df_test[i].fillna(df_test[i].mode()[0])

In [175]:
# Train

In [176]:
X_train = df_train.drop(columns="SalePrice")
y_train = df_train["SalePrice"]

In [177]:
y_train_log = np.log1p(y_train)

In [178]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train_log, test_size=0.2, random_state=42)

In [179]:
preprocessor = ColumnTransformer(
    [("cat", OneHotEncoder(sparse_output=False, drop="if_binary",
                          handle_unknown="infrequent_if_exist"), lapse_cat),
     ("ord", OrdinalEncoder(categories=ordinal_categories), lapse_ord),
     ("int", StandardScaler(), lapse_int)
    ])

preprocessor.set_output(transform="pandas") 

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('ord', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e

In [180]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

In [181]:
models = {
    "GradBoost": GradientBoostingRegressor(
        n_estimators=150,
        learning_rate=0.02,
        max_depth=2,
        random_state=42
    ),

    "CatBoost": CatBoostRegressor(
        iterations=250,
        loss_function="RMSE",
        learning_rate=0.02,
        depth=3,
        l2_leaf_reg=3,
        verbose=0,
    ),
    
    # "LGBM": LGBMClassifier(n_estimators=250,
    #     num_leaves=5,
    #     max_depth=2,
    #     min_child_samples=2,
    #     min_gain_to_split=0.01,
    #     reg_alpha=10,
    #     reg_lambda=5,
    #     ),
    
    "XGB": XGBRegressor(n_estimators=256,
      learning_rate=0.03,
      max_depth=3,
      min_child_weight=5,
      gamma=0.05,
      subsample=0.85,
      colsample_bytree=0.85,
      reg_alpha=0.1,
      reg_lambda=5,
      objective="reg:squarederror",
      eval_metric="rmse",
      random_state=42,
      n_jobs=-1),
}

In [182]:
res_mean = {}
res_std = {}

In [183]:
for name, reg in models.items():
    pipe = Pipeline(
        [('prepocessor', preprocessor), ("regressor", reg)])
    scores = cross_val_score(
        pipe,
        X_tr,
        y_tr,
        cv=cv,
        scoring="neg_root_mean_squared_error"
    )
    scores = -scores
    res_mean[name] = scores.mean()
    res_std[name] = scores.std()
    print(
        name,
        scores.mean(),
        scores.std()
    )

/home/sergey/venvs/py/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [5] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
/home/sergey/venvs/py/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [6] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
/home/sergey/venvs/py/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [5] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
/home/sergey/venvs/py/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 5, 6] during transform. These unknown categories will be enco

GradBoost 0.19860364179138232 0.017578349228004407


/home/sergey/venvs/py/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [6] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
/home/sergey/venvs/py/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [5] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
/home/sergey/venvs/py/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 5, 6] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)


CatBoost 0.19076442774217023 0.015990424762239608


/home/sergey/venvs/py/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [5] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
/home/sergey/venvs/py/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [6] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
/home/sergey/venvs/py/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [5] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)


XGB 0.18332464875538645 0.013768446416264953


/home/sergey/venvs/py/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 5, 6] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)


In [184]:
best_model_name = min(res_mean, key=res_mean.get)

print(f"Название лучшей модели: {best_model_name}", f"CV скор: {res_mean[best_model_name]}", sep="\n")

Название лучшей модели: XGB
CV скор: 0.18332464875538645


In [185]:
best_reg = models[best_model_name]

best_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', best_reg)
])

best_pipe.fit(X_tr, y_tr)

val_pred = best_pipe.predict(X_val)

val_score = root_mean_squared_error(y_val, val_pred)

print(f"{best_model_name}: Hold-out RMSE:", val_score)
print(f"{best_model_name}: CV RMSE:", res_mean[best_model_name], res_std[best_model_name])

XGB: Hold-out RMSE: 0.1951802421520191
XGB: CV RMSE: 0.18332464875538645 0.013768446416264953


/home/sergey/venvs/py/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
